# DDPM walkthrough\n
This notebook loads a trained checkpoint, visualizes forward noising, and generates an EMA sample grid.

In [ ]:
from pathlib import Path\n
import sys, torch\n
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()\n
sys.path.insert(0, str(ROOT))\n
from models import EMA, GaussianDiffusion\n
from training.train import build_dataloader, build_model\n
from utils.config import load_config\n
from utils.visualization import save_noising_steps, save_sample_grid\n

In [ ]:
config = load_config(ROOT / 'configs/cifar10.yaml')
checkpoint_path = ROOT / 'outputs/cifar10/checkpoints/epoch_030.pt'  # Change after training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
state = torch.load(checkpoint_path, map_location=device, weights_only=False)
model = build_model(config).to(device)
ema = EMA(model, config['ema_decay'])
ema.load_state_dict(state['ema'])
diffusion = GaussianDiffusion(config['timesteps'], config['beta_schedule']).to(device)

In [ ]:
images, _ = next(iter(build_dataloader(config)))
save_noising_steps(diffusion, images[:1].to(device), ROOT / 'outputs/cifar10/notebook_noising.png')
from IPython.display import Image
Image(filename=str(ROOT / 'outputs/cifar10/notebook_noising.png'))

In [ ]:
shape = (64, config['image_channels'], config['image_size'], config['image_size'])
samples = diffusion.sample(ema.ema_model.to(device), shape, device)
save_sample_grid(samples, ROOT / 'outputs/cifar10/notebook_samples.png')
Image(filename=str(ROOT / 'outputs/cifar10/notebook_samples.png'))